In [ ]:
import os
import shutil
import ast
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from base64 import b64encode
from IPython.display import HTML
from ultralytics import YOLO

## Helper functions

In [ ]:
def process_debris_all(csv_file, src_img_dir, dest_img_dir, dest_label_dir):
    os.makedirs(dest_img_dir, exist_ok=True)
    os.makedirs(dest_label_dir, exist_ok=True)
    if not os.path.exists(csv_file): return

    df = pd.read_csv(csv_file)
    for _, row in df.iterrows():
        image_id = row['ImageID']
        img_name = f"{image_id}.jpg"
        src_img_path = os.path.join(src_img_dir, img_name)
        
        if not os.path.exists(src_img_path): continue
        shutil.copy(src_img_path, os.path.join(dest_img_dir, img_name))
        
        with Image.open(src_img_path) as img: 
            img_w, img_h = img.size
            
        try: boxes = ast.literal_eval(row['bboxes'])
        except: boxes = []
            
        with open(os.path.join(dest_label_dir, f"{image_id}.txt"), "w") as f_label:
            for box in boxes:
                xmin, xmax, ymin, ymax = map(float, box)
                w, h = xmax - xmin, ymax - ymin
                x_center, y_center = xmin + (w / 2.0), ymin + (h / 2.0)
                f_label.write(f"0 {x_center/img_w:.6f} {y_center/img_h:.6f} {w/img_w:.6f} {h/img_h:.6f}\n")

def copy_earth_data(src_base, split, dest_base):
    src_images_dir, src_labels_dir = os.path.join(src_base, split, 'images'), os.path.join(src_base, split, 'labels')
    dest_images_dir, dest_labels_dir = os.path.join(dest_base, split, 'images'), os.path.join(dest_base, split, 'labels')
    
    os.makedirs(dest_images_dir, exist_ok=True)
    os.makedirs(dest_labels_dir, exist_ok=True)
    if not os.path.exists(src_images_dir): return
        
    all_images = [f for f in os.listdir(src_images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    for img_name in all_images:
        shutil.copy(os.path.join(src_images_dir, img_name), os.path.join(dest_images_dir, img_name))
        txt_name = os.path.splitext(img_name)[0] + '.txt'
        src_txt_path = os.path.join(src_labels_dir, txt_name)
        
        if os.path.exists(src_txt_path): 
            shutil.copy(src_txt_path, os.path.join(dest_labels_dir, txt_name))
        else: 
            open(os.path.join(dest_labels_dir, txt_name), 'w').close()

def plot_comparison(img_path, label_path, model_baseline, model_ext, conf_thresh=0.45, output_filename='comparison.png'):
    res_base = model_baseline(img_path, conf=conf_thresh, verbose=False)[0].plot()[..., ::-1]
    res_ext = model_ext(img_path, conf=conf_thresh, verbose=False)[0].plot()[..., ::-1]

    img_gt = Image.open(img_path)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(img_gt)
    axes[0].set_title('Ground truth', fontsize=14, fontweight='bold')
    axes[0].axis('off')

    img_w, img_h = img_gt.size
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id, x_center, y_center, width, height = map(float, parts[:5])
                    abs_w, abs_h = width * img_w, height * img_h
                    xmin, ymin = (x_center * img_w) - (abs_w / 2.0), (y_center * img_h) - (abs_h / 2.0)
                    rect = patches.Rectangle((xmin, ymin), abs_w, abs_h, linewidth=3, edgecolor='lime', facecolor='none')
                    axes[0].add_patch(rect)
    else:
        print(f"Warning: Label file not found at {label_path}")

    axes[1].imshow(res_base)
    axes[1].set_title('YOLO26n baseline', fontsize=14, fontweight='bold')
    axes[1].axis('off')

    axes[2].imshow(res_ext)
    axes[2].set_title('YOLO26n extended dataset', fontsize=14, fontweight='bold')
    axes[2].axis('off')

    plt.tight_layout()
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.show()

## Paths and model loading

In [ ]:
BASE_DEBRIS_DIR = '/kaggle/input/datasets/sadianawar/debris-detection-dataset/debris-detection'
EXT_EARTH_DIR = '/kaggle/input/datasets/ewakasprzak/earth-satellite-images-512x512/earth-satellite-images-dataset'

model_baseline = YOLO('/kaggle/input/models/ewakasprzak/space-debris-yolo26n/pytorch/space-debris/1/best.pt')
model_ext = YOLO('/kaggle/input/models/ewakasprzak/space-debris-yolo26n-extended/pytorch/space-debris/1/best.pt')

## Evaluate on debris dataset

In [ ]:
YOLO_DIR_EXP = "/kaggle/working/zorza_dataset"
process_debris_all(
    os.path.join(BASE_DEBRIS_DIR, "val.csv"), 
    os.path.join(BASE_DEBRIS_DIR, "val"), 
    os.path.join(YOLO_DIR_EXP, "val", "images"), 
    os.path.join(YOLO_DIR_EXP, "val", "labels")
)

test_image_id = '1036'
img_path = os.path.join(YOLO_DIR_EXP, "val", "images", f"{test_image_id}.jpg")
label_path = os.path.join(YOLO_DIR_EXP, "val", "labels", f"{test_image_id}.txt")

plot_comparison(img_path, label_path, model_baseline, model_ext, output_filename='comparison_debris.png')

## Evaluate on Earth dataset

In [ ]:
YOLO_DIR_EARTH = "/kaggle/working/zorza_dataset_earth"
copy_earth_data(EXT_EARTH_DIR, 'val', YOLO_DIR_EARTH)

test_image_id = 'bg_val_00031'
img_path = os.path.join(YOLO_DIR_EARTH, "val", "images", f"{test_image_id}.jpg")
label_path = os.path.join(YOLO_DIR_EARTH, "val", "labels", f"{test_image_id}.txt")

plot_comparison(img_path, label_path, model_baseline, model_ext, output_filename='comparison_earth.png')

## Video tracking

In [ ]:
input_video = '/kaggle/input/datasets/ewakasprzak/space-debris-collision/space-debris-collision.mp4'
output_dir = '/kaggle/working/video_results'

print("Processing video frames...")
model_ext.predict(
    source=input_video, 
    conf=0.50, 
    save=True, 
    project=output_dir, 
    name='inference'
)

base_name = os.path.basename(input_video).rsplit('.', 1)[0]
avi_path = f"{output_dir}/inference/{base_name}.avi"
mp4_path = f"{output_dir}/inference/{base_name}_web.mp4"

print("Converting AVI to MP4 for web playback...")
!ffmpeg -y -i "{avi_path}" -vcodec libx264 "{mp4_path}" -hide_banner -loglevel error

print("Loading video player...")
mp4_data = open(mp4_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4_data).decode()

HTML(f"""
<video width=800 controls autoplay loop>
      <source src="{data_url}" type="video/mp4">
      Your browser does not support the video tag.
</video>
""")